In [1]:
import argparse
import pickle
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

In [12]:
data = np.load("curated_airfoils.npz", allow_pickle=True)


In [14]:
shapes = data['shapes']
classes = data['classes']

In [15]:
def extract_all_features(shapes):
    """
    Return 18 geometric features from a (1001, 2) airfoil coordinate array.
 
    Features
    --------
    [0]  max_thickness          maximum thickness / chord
    [1]  x_max_thickness        chord-wise location of max thickness
    [2]  max_camber             maximum |camber| / chord
    [3]  x_max_camber           chord-wise location of max camber
    [4]  mean_camber            mean camber / chord
    [5]  te_thickness           trailing-edge thickness
    [6]  le_thickness           leading-edge thickness proxy (at 1 % chord)
    [7]  le_camber_slope        d(camber)/dx at leading edge
    [8]  te_camber_slope        d(camber)/dx at trailing edge
    [9-14]  thickness at x = 12, 25, 40, 60, 75, 90 %
    [15-17] camber   at x = 25, 50, 75 %
    """
    x = shapes[:,0]
    y = shapes[:,1]
    le_idx = int(np.argmin(x))
    
    # split upper / lower and orient from LE → TE
    x_up = x[:le_idx + 1][::-1]
    y_up = y[:le_idx + 1][::-1]
    x_lo = x[le_idx:]
    y_lo = y[le_idx:]
    
    xc = np.linspace(0, 1, 200)
    try:
        yu = np.interp(xc, x_up, y_up)
        yl = np.interp(xc, x_lo, y_lo)
    except Exception:
        return None
 
    thick  = yu - yl
    camber = (yu + yl) / 2.0
 
    max_t    = float(np.max(thick))
    x_max_t  = float(xc[np.argmax(thick)])
    max_c    = float(np.max(np.abs(camber)))
    x_max_c  = float(xc[np.argmax(np.abs(camber))]) if max_c > 1e-8 else 0.5
    mean_c   = float(np.mean(camber))
    te_t     = float(thick[-1])
    le_t     = float(thick[2])           # ≈ 1 % chord
 
    dcamber  = np.gradient(camber, xc)
    le_slope = float(dcamber[2])
    te_slope = float(dcamber[-3])
 
    t_stations = np.interp([0.12, 0.25, 0.40, 0.60, 0.75, 0.90], xc, thick)
    c_stations = np.interp([0.25, 0.50, 0.75],                    xc, camber)
 
    feat = np.array([max_t, x_max_t, max_c, x_max_c, mean_c,
                     te_t, le_t, le_slope, te_slope,
                     *t_stations, *c_stations])
 
    if np.any(np.isnan(feat)) or np.any(np.isinf(feat)):
        return None
    return feat
    

In [19]:
FEATURE_NAMES = [
    "max_thickness", "x_max_thickness", "max_camber", "x_max_camber",
    "mean_camber", "te_thickness", "le_thickness",
    "le_camber_slope", "te_camber_slope",
    "t@12%", "t@25%", "t@40%", "t@60%", "t@75%", "t@90%",
    "c@25%", "c@50%", "c@75%",
]

print("Extracting geometric features …")
features_list, valid_idx = [], []
for i, shape in enumerate(shapes):
    f = extract_all_features(shape)
    if f is not None:
        features_list.append(f)
        valid_idx.append(i)
        
features    = np.array(features_list)           # (N, 18)
valid_idx   = np.array(valid_idx)
valid_cls   = classes[valid_idx]
print(f"  Valid airfoils : {len(features):,}")
 
# convenience aliases
max_t  = features[:, 0]
max_c  = features[:, 2]
mean_c = features[:, 4]

Extracting geometric features …
  Valid airfoils : 19,164


PHYSICS-BASED Cl / Cd GENERATION
Cl  — thin airfoil theory + Helmbold thickness correction + Kirchhoff flow-separation stall model 
Cd  — empirical drag polar  Cd = Cd0 + K·Cl² + post-stall drag surge (sinusoidal)

In [21]:
print("Generating physics-based Cl / Cd …")
 
alpha_deg = np.linspace(-10, 25, 36)   # sweep −10 … +25 °
alpha_rad = np.deg2rad(alpha_deg)
 
# --- Lift ---
alpha_L0      = -(2.0 * mean_c + 0.5 * features[:, 15] * 0.5)   # zero-lift angle (rad)
thick_corr    = 1.0 + 0.77 * max_t                                # Helmbold correction
Cl_alpha_slp  = 2.0 * np.pi * thick_corr                          # Cl_α  (per radian)
alpha_stall   = np.clip(8 + 20 * features[:, 6] + 5 * max_t
                        - 3 * max_c, 8, 18)                       # stall angle (deg)
Cl_max_arr    = 1.0 + 2.0 * max_c + 0.3 * max_t

Generating physics-based Cl / Cd …


In [22]:
def kirchhoff_cl(alpha_d, stall_d, Cl_max, Cl_alpha, al0):
    """Kirchhoff-based Cl with separation model (vectorised over alpha)."""
    ar  = np.deg2rad(alpha_d)
    asr = np.deg2rad(stall_d)
    Cl_lin = Cl_alpha * (ar - al0)
 
    delta = ar - asr
    f     = np.where(delta <= 0, 1.0,
                     np.exp(-2.5 * delta / (0.1 + 0.3 * asr)))
    f     = np.clip(f, 0, 1)
    Cl_k  = Cl_max * ((1 + np.sqrt(f)) / 2) ** 2
 
    blend = np.where(delta <= 0, 1.0, np.exp(-4.0 * np.clip(delta, 0, None)))
    Cl    = blend * Cl_lin + (1 - blend) * Cl_k
 
    # negative stall mirror
    neg_mask = alpha_d < -stall_d
    f_neg    = np.exp(-2.5 * np.abs(np.minimum(delta, 0)) / (0.1 + 0.3 * asr))
    Cl_neg   = -Cl_max * ((1 + np.sqrt(np.clip(f_neg, 0, 1))) / 2) ** 2
    Cl       = np.where(neg_mask, Cl_neg, Cl)
    return Cl